In [141]:
import numpy as np
import pandas as pd

In [142]:
data = pd.read_csv('breast-cancer.csv')

# convert labels
data['diagnosis'] = data['diagnosis'].map({'B': 0, 'M': 1})

# convert to numpy
data = np.array(data)

# shuffle entire dataset
np.random.shuffle(data)

# split
train_data = data[:250]
test_data = data[250:]

# transpose for NN format (features, samples)
df_train = train_data.T
df_test = test_data.T

# separate X and Y
Y_train = df_train[1:2]
X_train = df_train[2:]

Y_test = df_test[1:2]
X_test = df_test[2:]

X_train = X_train / np.max(X_train)
X_test = X_test / np.max(X_test)

In [143]:
print(X_train.shape)

(30, 250)


In [144]:
print(data.shape , df_train.shape)

(569, 32) (32, 250)


In [145]:
def init_params():
    W1 = np.random.rand(16,30) - 0.5
    b1 = np.random.rand(16,1) - 0.5
    W2 = np.random.rand(8,16) - 0.5
    b2 = np.random.rand(8,1) - 0.5
    W3 = np.random.rand(1,8) - 0.5
    b3 = np.random.rand(1,1) - 0.5
    return W1,b1,W2,b2,W3,b3

def ReLu(Z):
    return np.maximum(0,Z)

def tanh(Z):
    x = np.exp(2*Z)
    return (x-1)/(1+x)

def sigmoid(Z):
    return 1/(1+np.exp(-Z))

def fwd_propogate(W1,b1,W2,b2,W3,b3,X):
    Z1 = W1.dot(X) + b1
    A1 = ReLu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = tanh(Z2)
    Z3 = W3.dot(A2) + b3
    A3 = sigmoid(Z3)
    return Z1 , A1 , Z2 , A2 , Z3 , A3

def backward_propagate(Z1, A1, Z2, A2, Z3, A3, W2, W3, X, Y):
    m = X.shape[1]

    # --- Output layer ---
    dZ3 = A3 - Y                      # (1, m)
    dW3 = (1/m) * dZ3.dot(A2.T)      # (1,16)
    db3 = (1/m) * np.sum(dZ3, axis=1, keepdims=True)

    # --- Layer 2 ---
    dA2 = W3.T.dot(dZ3)              # (16, m)
    dZ2 = dA2 * (1 - np.power(A2, 2))  # tanh derivative
    dW2 = (1/m) * dZ2.dot(A1.T)      # (16,8)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)

    # --- Layer 1 ---
    dA1 = W2.T.dot(dZ2)              # (8, m)
    dZ1 = dA1 * (Z1 > 0)             # ReLU derivative
    dW1 = (1/m) * dZ1.dot(X.T)       # (8,32)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)

    return dW1, db1, dW2, db2, dW3, db3

def update_params(W1, b1, W2, b2, W3, b3, dW1, db1, dW2, db2,dW3,db3, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    W3 = W3 - alpha * dW3
    b3 = b3 - alpha * db3
    return W1, b1, W2, b2, W3, b3



In [146]:
def get_predictions(A3):
 return (A3>=0.5).astype(int)

def get_accuracy(predictions, Y):

    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 , W3 , b3 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2,Z3,A3 = fwd_propogate(W1, b1, W2, b2,W3,b3, X)
        dW1, db1, dW2, db2,dW3,db3 = backward_propagate(Z1, A1, Z2, A2,Z3,A3, W2,W3 , X, Y)
        W1, b1, W2, b2, W3 , b3 = update_params(W1, b1, W2, b2,W3,b3, dW1, db1, dW2, db2,dW3,db3, alpha)
        if i % 100 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A3)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2 ,W3 , b3

In [147]:
W1 , b1 , W2 , b2 , W3, b3 = gradient_descent(X_train,Y_train,0.1,5000)

Iteration:  0
0.392
Iteration:  100
0.828
Iteration:  200
0.896
Iteration:  300
0.904
Iteration:  400
0.9
Iteration:  500
0.9
Iteration:  600
0.908
Iteration:  700
0.908
Iteration:  800
0.912
Iteration:  900
0.912
Iteration:  1000
0.912
Iteration:  1100
0.912
Iteration:  1200
0.912
Iteration:  1300
0.916
Iteration:  1400
0.916
Iteration:  1500
0.916
Iteration:  1600
0.916
Iteration:  1700
0.916
Iteration:  1800
0.916
Iteration:  1900
0.916
Iteration:  2000
0.916
Iteration:  2100
0.916
Iteration:  2200
0.92
Iteration:  2300
0.92
Iteration:  2400
0.92
Iteration:  2500
0.92
Iteration:  2600
0.92
Iteration:  2700
0.92
Iteration:  2800
0.916
Iteration:  2900
0.916
Iteration:  3000
0.912
Iteration:  3100
0.912
Iteration:  3200
0.908
Iteration:  3300
0.908
Iteration:  3400
0.908
Iteration:  3500
0.912
Iteration:  3600
0.912
Iteration:  3700
0.912
Iteration:  3800
0.912
Iteration:  3900
0.912
Iteration:  4000
0.912
Iteration:  4100
0.912
Iteration:  4200
0.912
Iteration:  4300
0.912
Iteration:

In [148]:
def make_predictions(X, W1, b1, W2, b2 , W3, b3):
    _, _, _, _,_,A3 = fwd_propogate(W1, b1, W2, b2,W3,b3, X)
    predictions = get_predictions(A3)
    return predictions

def test_prediction(index, W1, b1, W2, b2, W3, b3):
    current_image = X_train[:, index, None]
    prediction = make_predictions(X_train[:, index, None], W1, b1, W2, b2,W3,b3)
    label = Y_train[0,index]
    print("Prediction: ", prediction)
    print("Label: ", label)

In [149]:
test_prediction(0,W1,b1,W2,b2,W3,b3)
test_prediction(101,W1,b1,W2,b2,W3,b3)
test_prediction(20,W1,b1,W2,b2,W3,b3)
test_prediction(37,W1,b1,W2,b2,W3,b3)
test_prediction(198,W1,b1,W2,b2,W3,b3)
test_prediction(23,W1,b1,W2,b2,W3,b3)
test_prediction(78,W1,b1,W2,b2,W3,b3)
test_prediction(92,W1,b1,W2,b2,W3,b3)
test_prediction(80,W1,b1,W2,b2,W3,b3)

Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[1]]
Label:  1.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0
Prediction:  [[0]]
Label:  0.0


In [150]:
test_predictions = make_predictions(
    X_test,W1,b1,W2,b2,W3,b3
)

print(
    get_accuracy(test_predictions,Y_test)
)

0.8746081504702194


In [151]:
train_predictions = make_predictions(
    X_train,W1,b1,W2,b2,W3,b3
)

print(
    "Train Accuracy:",
    get_accuracy(train_predictions,Y_train)
)

print(
    "Test Accuracy:",
    get_accuracy(test_predictions,Y_test)
)

Train Accuracy: 0.912
Test Accuracy: 0.8746081504702194
